# 01 — KADID-10k data check (B-03)

Counts, MOS distribution, score direction. Saves figures to `results/figures/`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from humper.data import REPO_ROOT, load_image, load_pairs

FIG = REPO_ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
pairs = load_pairs()
pairs.head()

## Counts
Expect 81 references × 25 distortion types × 5 levels = 10,125 pairs.

In [ ]:
print(len(pairs), "pairs |", pairs.ref_id.nunique(), "refs |", pairs.dist_type.nunique(), "types |", pairs.level.nunique(), "levels")
assert len(pairs) == 81 * 25 * 5
missing = [p for p in pd.concat([pairs.ref_path, pairs.dist_path]).unique() if not (REPO_ROOT / p).exists()]
print("missing files:", len(missing))
img = load_image(pairs.dist_path[0]); print("image tensor:", tuple(img.shape), float(img.min()), float(img.max()))

## MOS distribution and direction
KADID's `dmos` column is on a 1–5 scale where **higher = better** (despite the name). If that holds, MOS should fall as distortion level rises.

In [ ]:
print(pairs.mos.describe())
by_level = pairs.groupby("level").mos.mean()
print(by_level)
assert by_level.is_monotonic_decreasing, "MOS should decrease with distortion level -> higher = better"

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
pairs.mos.hist(bins=40, ax=ax[0]); ax[0].set(xlabel="MOS (higher = better)", ylabel="count", title="MOS histogram")
pairs.boxplot(column="mos", by="level", ax=ax[1]); ax[1].set(title="MOS by distortion level", xlabel="level"); fig.suptitle("")
fig.tight_layout(); fig.savefig(FIG / "mos_distribution.png", dpi=150)

## Per distortion type

In [ ]:
per_type = (pairs.groupby(["dist_group", "dist_type", "dist_name"]).mos
            .agg(["count", "mean", "std", "min", "max"]).round(2))
per_type